In [1]:
import requests
import io
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.metrics import accuracy_score, f1_score


DATASET_API_ENDPOINT = (
    "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
)


# =====================================================================
# TASK 1 — API FETCHING FUNCTION
# =====================================================================

def fetch_triage_data(url: str) -> pd.DataFrame:
    """
    Fetch the diagnostic CSV dataset from the REST endpoint.

    Steps:
    1. Send an HTTP GET request.
    2. Raise an exception for HTTP errors.
    3. Read the response text through io.StringIO.
    4. Convert the CSV into a Pandas DataFrame.
    """
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()

        csv_buffer = io.StringIO(response.text)
        dataframe = pd.read_csv(csv_buffer)

        return dataframe

    except requests.exceptions.RequestException as error:
        raise RuntimeError(
            f"Failed to fetch triage data from API: {error}"
        ) from error

    except pd.errors.ParserError as error:
        raise RuntimeError(
            f"Failed to parse API CSV response: {error}"
        ) from error


# =====================================================================
# PIPELINE
# =====================================================================

def run_pipeline():
    # ---------------------------------------------------------------
    # TASK 1 — Telemetry API Fetch & Scaling
    # ---------------------------------------------------------------

    raw_df = fetch_triage_data(DATASET_API_ENDPOINT)

    # Remove records containing missing values.
    df = raw_df.dropna().copy()

    # Target:
    # Chinstrap -> High Urgency (1)
    # Everything else -> Low Urgency (0)
    df["High_Urgency"] = (
        df["species"].astype(str).str.strip().str.lower() == "chinstrap"
    ).astype(int)

    # The penguins dataset contains four continuous numeric measurements.
    feature_columns = [
        "bill_length_mm",
        "bill_depth_mm",
        "flipper_length_mm",
        "body_mass_g",
    ]

    X = df[feature_columns].copy()
    y = df["High_Urgency"].copy()

    # Stratified 75/25 split.
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.25,
        stratify=y,
        random_state=42,
    )

    # Fit scaler ONLY on training data to avoid test-data leakage.
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    print("[TASK 1 SUCCESS] Triage Data Ingested")
    print(f"Clean Records: {len(df)}")
    print(
        f"Train Set: {len(X_train)} Records | "
        f"Test Set: {len(X_test)} Records"
    )
    print("Target Counts (0: Low / 1: High):")
    print(y.value_counts().sort_index())

    # ---------------------------------------------------------------
    # TASK 2 — Base Models & Soft Voting
    # ---------------------------------------------------------------

    lr = LogisticRegression()
    knn = KNeighborsClassifier(n_neighbors=5)
    dt = DecisionTreeClassifier(max_depth=4, random_state=42)

    base_estimators = [
        ("lr", lr),
        ("knn", knn),
        ("dt", dt),
    ]

    voting_classifier = VotingClassifier(
        estimators=base_estimators,
        voting="soft",
    )

    # Train individual base models.
    lr.fit(X_train_scaled, y_train)
    knn.fit(X_train_scaled, y_train)
    dt.fit(X_train_scaled, y_train)

    lr_predictions = lr.predict(X_test_scaled)
    knn_predictions = knn.predict(X_test_scaled)
    dt_predictions = dt.predict(X_test_scaled)

    lr_accuracy = accuracy_score(y_test, lr_predictions)
    knn_accuracy = accuracy_score(y_test, knn_predictions)
    dt_accuracy = accuracy_score(y_test, dt_predictions)

    # Train soft voting ensemble.
    voting_classifier.fit(X_train_scaled, y_train)
    voting_predictions = voting_classifier.predict(X_test_scaled)

    voting_accuracy = accuracy_score(y_test, voting_predictions)

    print("\n[TASK 2 SUCCESS] Soft Voting Ensemble Trained:")
    print(
        f"Base Model 1 (Logistic Regression) Accuracy : "
        f"{lr_accuracy:.2%}"
    )
    print(
        f"Base Model 2 (K-Nearest Neighbors) Accuracy : "
        f"{knn_accuracy:.2%}"
    )
    print(
        f"Base Model 3 (Decision Tree) Accuracy       : "
        f"{dt_accuracy:.2%}"
    )
    print(
        f"Ensemble Soft Voting Test Accuracy          : "
        f"{voting_accuracy:.2%}"
    )

    # ---------------------------------------------------------------
    # TASK 3 — Stacking Classifier
    # ---------------------------------------------------------------

    stacking_estimators = [
        (
            "lr",
            LogisticRegression(),
        ),
        (
            "knn",
            KNeighborsClassifier(n_neighbors=5),
        ),
        (
            "dt",
            DecisionTreeClassifier(
                max_depth=4,
                random_state=42,
            ),
        ),
    ]

    stacking_classifier = StackingClassifier(
        estimators=stacking_estimators,
        final_estimator=LogisticRegression(),
    )

    stacking_classifier.fit(X_train_scaled, y_train)

    stacking_predictions = stacking_classifier.predict(X_test_scaled)

    stacking_accuracy = accuracy_score(
        y_test,
        stacking_predictions,
    )

    stacking_f1 = f1_score(
        y_test,
        stacking_predictions,
        zero_division=0,
    )

    # F1 scores for the individual models and voting ensemble.
    lr_f1 = f1_score(y_test, lr_predictions, zero_division=0)
    knn_f1 = f1_score(y_test, knn_predictions, zero_division=0)
    dt_f1 = f1_score(y_test, dt_predictions, zero_division=0)
    voting_f1 = f1_score(
        y_test,
        voting_predictions,
        zero_division=0,
    )

    print("\n[TASK 3 SUCCESS] Stacking Classifier Trained:")
    print(
        f"Stacking Meta-Learner Test Accuracy : "
        f"{stacking_accuracy:.2%}"
    )
    print(
        f"Stacking Meta-Learner Test F1-Score : "
        f"{stacking_f1:.4f}"
    )

    # ---------------------------------------------------------------
    # FINAL SUMMARY
    # ---------------------------------------------------------------

    print(
        "\n========== HETEROGENEOUS VOTING & STACKING ENSEMBLE ENGINE =========="
    )

    print(
        "\nData Ingestion Status      : "
        "REST API Ingestion Successful (HTTP 200 OK)"
    )
    print(
        f"Master Dataset Records     : {len(df)} Diagnostic Records"
    )
    print(
        "Ensemble Diversity         : "
        "Linear (Logistic Regression), "
        "Distance-Based (k-NN), "
        "Tree-Based (Decision Tree)"
    )

    print("\nArchitecture Performance Benchmark:")
    print(
        "+------------------------------------+---------------+------------+"
    )
    print(
        "| Model Architecture                 | Test Accuracy | F1-Score   |"
    )
    print(
        "+------------------------------------+---------------+------------+"
    )
    print(
        f"| Individual Logistic Regression     | "
        f"{lr_accuracy:>12.2%} | {lr_f1:>10.4f} |"
    )
    print(
        f"| Individual K-Nearest Neighbors     | "
        f"{knn_accuracy:>12.2%} | {knn_f1:>10.4f} |"
    )
    print(
        f"| Individual Decision Tree           | "
        f"{dt_accuracy:>12.2%} | {dt_f1:>10.4f} |"
    )
    print(
        f"| Ensemble Soft Voting Classifier    | "
        f"{voting_accuracy:>12.2%} | {voting_f1:>10.4f} |"
    )
    print(
        f"| Stacking Classifier (Meta-Learner) | "
        f"{stacking_accuracy:>12.2%} | {stacking_f1:>10.4f} |"
    )
    print(
        "+------------------------------------+---------------+------------+"
    )

    print(
        "\nConclusion:"
        "\nStacking combines predictions from heterogeneous base algorithms "
        "and trains a Logistic Regression meta-learner on those predictions. "
        "The measured test metrics above are computed from the live dataset "
        "rather than hard-coded."
    )

    return {
        "dataset_records": len(df),
        "train_records": len(X_train),
        "test_records": len(X_test),
        "target_counts": y.value_counts().sort_index().to_dict(),
        "logistic_regression": {
            "accuracy": lr_accuracy,
            "f1": lr_f1,
        },
        "knn": {
            "accuracy": knn_accuracy,
            "f1": knn_f1,
        },
        "decision_tree": {
            "accuracy": dt_accuracy,
            "f1": dt_f1,
        },
        "soft_voting": {
            "accuracy": voting_accuracy,
            "f1": voting_f1,
        },
        "stacking": {
            "accuracy": stacking_accuracy,
            "f1": stacking_f1,
        },
    }


# =====================================================================
# MAIN
# =====================================================================

if __name__ == "__main__":
    results = run_pipeline()


[TASK 1 SUCCESS] Triage Data Ingested
Clean Records: 333
Train Set: 249 Records | Test Set: 84 Records
Target Counts (0: Low / 1: High):
High_Urgency
0    265
1     68
Name: count, dtype: int64

[TASK 2 SUCCESS] Soft Voting Ensemble Trained:
Base Model 1 (Logistic Regression) Accuracy : 96.43%
Base Model 2 (K-Nearest Neighbors) Accuracy : 96.43%
Base Model 3 (Decision Tree) Accuracy       : 94.05%
Ensemble Soft Voting Test Accuracy          : 96.43%

[TASK 3 SUCCESS] Stacking Classifier Trained:
Stacking Meta-Learner Test Accuracy : 97.62%
Stacking Meta-Learner Test F1-Score : 0.9375

========== HETEROGENEOUS VOTING & STACKING ENSEMBLE ENGINE ==========

Data Ingestion Status      : REST API Ingestion Successful (HTTP 200 OK)
Master Dataset Records     : 333 Diagnostic Records
Ensemble Diversity         : Linear (Logistic Regression), Distance-Based (k-NN), Tree-Based (Decision Tree)

Architecture Performance Benchmark:
+------------------------------------+---------------+------------